# Training a SOM species classifier

Trains and evaluates the SOM classifier from `FlowerEvolver-WASM`'s Python bindings, then exports the result for embedding as `fe::resources::trainedSOM`.

This is a **development tool**, not part of the runtime -- see `tools/README.md`. Run cells top to bottom; re-run the grid-size/training cells as many times as you like while tuning. `tools/som.py` does the same from the command line.

## Setup

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.insert(0, '.')  # so `import som_trainer` and `import som` find tools/ without installing anything

import flowerevolver as fe
import som  # tools/som.py: extract_samples() with the feature cache
from som_trainer import cache as feature_cache, dataset, evaluation, export, visualize

%matplotlib inline

## 1. Load your flowers

Point `GENERATIONS_JSON_PATH` at a real exported flower collection (the `{"Generation": [...]}` format `FlowerEvolver-frontend`'s exporter already produces -- see `dataset.load_generation_json()` for a single file, or `dataset.flowers_from_directory()` for a folder of individually-saved genomes).

No real collection on hand yet? `USE_SYNTHETIC_DATA = True` generates one with `make_flower()` so the rest of this notebook has something to run against. **Treat any classifier trained on synthetic data as a pipeline smoke test, not a real result** -- a random CPPN population isn't representative of what players actually grow. `tools/som.py --generate-dataset` makes a better-mixed one.

In [ ]:
USE_SYNTHETIC_DATA = False
GENERATIONS_JSON_PATH = 'favouritesFlowers.json'  # only used if USE_SYNTHETIC_DATA is False
SYNTHETIC_POPULATION_SIZE = 200

if USE_SYNTHETIC_DATA:
    flowers = dataset.generate_synthetic_population(SYNTHETIC_POPULATION_SIZE, seed=42)
    generations_json = dataset.generation_json_from_flowers(flowers)
else:
    generations_json = dataset.load_generation_json(GENERATIONS_JSON_PATH)

print(f'{dataset.count_generation_entries(generations_json)} flowers loaded')

## 2. Pick an environment

The environment `Stats` (and so the feature vector's last 11 dimensions) are computed under. The game varies this per-flower on purpose (weather) -- this is just what *this training run* uses; it's recorded on the resulting classifier for documentation only, not enforced at classify time (see `Environment`'s own docstring).

In [ ]:
environment = fe.Environment(humidity=0.5, temperature=20, altitude=0, terrain_type=0)
environment

## 3. Extract features

Turns every flower's genome into a 267-dim feature vector (16x16 silhouette + 11 normalized stats) once, up front -- every grid-size/training experiment below reuses these instead of re-rendering.

Rendering is the slow part (about 50 ms a flower), so the vectors are kept in a **feature cache** next to the dataset (`favouritesFlowers-cache.json`): the second run, or a run after you add flowers, only renders the new ones. The cache is keyed by flower content and by environment, and it never overwrites a file that is not a feature cache. Set `USE_CACHE = False` to render everything again.

In [ ]:
USE_CACHE = True
WORKERS = 0  # processes that render the flowers: 0 is one per core (see som_trainer/parallel.py)

cache = None
if USE_CACHE and not USE_SYNTHETIC_DATA:
    cache = feature_cache.FeatureCache(feature_cache.default_path(GENERATIONS_JSON_PATH), environment)

samples = som.extract_samples(generations_json, environment, cache, workers=WORKERS)
print(f'{len(samples)} usable flowers')

### Held-out flowers

A bigger grid always fits the flowers it was trained on better, because it can memorise them. So the grid-size comparison below also reports the error on flowers the map never saw: that is the number to trust. (The final training in section 6 uses every flower.)

In [ ]:
HOLDOUT_FRACTION = 0.25

shuffled = list(samples)
random.Random(42).shuffle(shuffled)
n_test = int(len(shuffled) * HOLDOUT_FRACTION)
test_samples, train_samples = shuffled[:n_test], shuffled[n_test:]
print(f'{len(train_samples)} to train on, {len(test_samples)} held out')

## 4. Explore grid sizes

**Don't skip this and hardcode a grid size** -- the right size depends on how varied this particular population actually is. Too small and visually distinct flowers get lumped together; too big and most nodes sit empty or duplicate their neighbors. Try a spread and look at the held-out error *and* the utilization heatmap in the next section before picking one -- a lower error from a bigger grid isn't a free win if half its nodes are unused.

`EXPLORATION_EPOCHS` is only for comparing sizes quickly; a longer run changes the numbers (the grid experiment in the todo used 150 epochs). The demo can fine-tune and split grids of up to 256 nodes (its WebAssembly heap is fixed), so larger ones are flagged.

In [ ]:
WASM_MAX_NODES = 256  # the demo's limit for training and fine-tuning in the browser

GRID_SIZES = [(8, 6), (8, 8), (10, 10), (12, 12), (16, 12), (16, 16)]
EXPLORATION_EPOCHS = 4

test_errors = {}
for width, height in GRID_SIZES:
    training = fe.SOMTrainingConfig(width, height, EXPLORATION_EPOCHS, base_lr=0.5, base_radius=max(width, height) / 2)
    classifier = fe.train_flower_classifier(train_samples, environment, training)
    train_error = evaluation.mean_quantization_error(classifier, train_samples)
    test_errors[(width, height)] = evaluation.mean_quantization_error(classifier, test_samples)
    note = '' if width * height <= WASM_MAX_NODES else '  (too big for the demo)'
    print(f'{width}x{height}: train {train_error:.4f}, held out {test_errors[(width, height)]:.4f}, '
          f'gap {test_errors[(width, height)] - train_error:.4f}{note}')

visualize.plot_grid_size_comparison(test_errors);

## 5. Check node utilization for a candidate size

Pick a `CANDIDATE_WIDTH`/`CANDIDATE_HEIGHT` from the previous section's output and sanity-check it here, on the held-out flowers, before committing to a full training run.

In [ ]:
CANDIDATE_WIDTH, CANDIDATE_HEIGHT = 12, 8
if CANDIDATE_WIDTH * CANDIDATE_HEIGHT > WASM_MAX_NODES:
    print(f'warning: {CANDIDATE_WIDTH * CANDIDATE_HEIGHT} nodes is more than the demo can fine-tune ({WASM_MAX_NODES})')

candidate_training = fe.SOMTrainingConfig(CANDIDATE_WIDTH, CANDIDATE_HEIGHT, EXPLORATION_EPOCHS,
                                          base_lr=0.5, base_radius=max(CANDIDATE_WIDTH, CANDIDATE_HEIGHT) / 2)
candidate_classifier = fe.train_flower_classifier(train_samples, environment, candidate_training)

print(evaluation.utilization_summary(candidate_classifier, test_samples))
visualize.plot_utilization_heatmap(evaluation.node_utilization(candidate_classifier, test_samples));

## 6. Validate weight precision on held-out flowers

This experiment uses a **separate validation SOM** trained only on `train_samples`. The held-out vectors are not used to fit its prototypes, so they provide a cleaner estimate of how rounding changes assignments away from the training data.

This is intentionally separate from the production model in the next section, which is trained on all usable flowers. We select a decimal precision here, then sanity-check that precision again on the production model before export.

The split is reproducible via `SPLIT_SEED`. With a small dataset, treat a difference of one or two flowers as noisy and try another seed before making a strong claim.


In [ ]:
# Reuse the same feature vectors and the existing grid-size split, but make the split seed explicit.
SPLIT_SEED = 42
HOLDOUT_FRACTION = 0.25
PRECISIONS = [6, 4, 3, 2, 1, 0]
MAX_CHANGED_RATE = 0.01

order = list(range(len(samples)))
random.Random(SPLIT_SEED).shuffle(order)
n_test = int(len(order) * HOLDOUT_FRACTION)
test_idx, train_idx = sorted(order[:n_test]), sorted(order[n_test:])
quant_train_samples = [samples[i] for i in train_idx]
quant_test_samples = [samples[i] for i in test_idx]

print(f"Precision validation: {len(quant_train_samples)} train, "
      f"{len(quant_test_samples)} held out")


### Train a reference map on the validation training split

The candidate grid comes from the earlier grid-size exploration. This reference model is not the production model; it exists to estimate the effect of decimal rounding on unseen flowers.


In [ ]:
validation_classifier = fe.FlowerClassifier(CANDIDATE_WIDTH, CANDIDATE_HEIGHT, environment)
VALIDATION_EPOCHS = 1500
VALIDATION_BASE_LR = 0.9
VALIDATION_BASE_RADIUS = max(CANDIDATE_WIDTH, CANDIDATE_HEIGHT) / 2
validation_report = validation_classifier.continue_training(
    quant_train_samples, VALIDATION_EPOCHS,
    VALIDATION_BASE_LR, VALIDATION_BASE_RADIUS
)
print(f"{len(validation_report.skipped)} validation-training flowers skipped")

for split_name, split_samples in (
    ("train", quant_train_samples),
    ("held out", quant_test_samples),
):
    qe = evaluation.mean_quantization_error(validation_classifier, split_samples)
    print(f"{split_name:9s} mean quantization error: {qe:.4f}")


### Compare rounded and exact weights

A changed assignment means that a flower's winning SOM coordinate changed. It does **not** imply that a species prediction changed unless you have a validated cell-to-species mapping. QE deltas are reported separately because assignments can remain stable while distances change.


In [ ]:
def _predict_cells(model, vectors):
    predictions = []
    for vector in vectors:
        match = model.classify_features(vector)
        predictions.append(
            None if match is None else (int(match.x), int(match.y), float(match.distance))
        )
    return predictions


def _quantized_copy(model, decimals):
    # Use the same serialization path as export, without mutating the reference model.
    model.weight_decimals = decimals
    try:
        serialized = model.to_json()
    finally:
        model.weight_decimals = None
    return fe.FlowerClassifier.from_json(serialized)


def _quantization_drift(reference_predictions, rounded_predictions, reference_model, rounded_model, vectors):
    changed = [
        i for i, (a, b) in enumerate(zip(reference_predictions, rounded_predictions))
        if (a is None) != (b is None) or (a is not None and a[:2] != b[:2])
    ]
    deltas = [
        b[2] - a[2] for a, b in zip(reference_predictions, rounded_predictions)
        if a is not None and b is not None
    ]
    qe_reference = evaluation.mean_quantization_error(reference_model, vectors)
    qe_rounded = evaluation.mean_quantization_error(rounded_model, vectors)
    return {
        "samples": len(vectors),
        "changed": changed,
        "changed_rate": len(changed) / len(vectors) if vectors else 0.0,
        "mean_abs_distance_delta": float(np.mean(np.abs(deltas))) if deltas else None,
        "qe_delta": qe_rounded - qe_reference,
    }


validation_splits = {
    "train": quant_train_samples,
    "held_out": quant_test_samples,
}
validation_reference_predictions = {
    name: _predict_cells(validation_classifier, vectors)
    for name, vectors in validation_splits.items()
}

precision_results = []
for decimals in PRECISIONS:
    rounded_model = _quantized_copy(validation_classifier, decimals)
    row = {"decimals": decimals}
    for split_name, vectors in validation_splits.items():
        row[split_name] = _quantization_drift(
            validation_reference_predictions[split_name],
            _predict_cells(rounded_model, vectors),
            validation_classifier, rounded_model, vectors
        )
    precision_results.append(row)

print(f"{'dec':>3} | {'train changed':>18} | {'held-out changed':>18} | {'held-out QE delta':>18}")
for row in precision_results:
    tr, ho = row["train"], row["held_out"]
    print(f"{row['decimals']:>3} | {len(tr['changed']):>4}/{tr['samples']:<4} "
          f"({tr['changed_rate']:>6.2%}) | {len(ho['changed']):>4}/{ho['samples']:<4} "
          f"({ho['changed_rate']:>6.2%}) | {ho['qe_delta']:>+18.4g}")


In [ ]:
labels = [str(row["decimals"]) for row in precision_results]
x = np.arange(len(precision_results))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(x - 0.2, [row["train"]["changed_rate"] * 100 for row in precision_results],
            0.4, label="validation train")
axes[0].bar(x + 0.2, [row["held_out"]["changed_rate"] * 100 for row in precision_results],
            0.4, label="held out")
axes[0].set_xticks(x, labels)
axes[0].set_xlabel("Weight decimal places")
axes[0].set_ylabel("Flowers changing SOM cell (%)")
axes[0].set_title("Assignment drift after rounding")
axes[0].legend()

axes[1].bar(x - 0.2, [row["train"]["qe_delta"] for row in precision_results],
            0.4, label="validation train")
axes[1].bar(x + 0.2, [row["held_out"]["qe_delta"] for row in precision_results],
            0.4, label="held out")
axes[1].axhline(0, linewidth=1, color="black")
axes[1].set_xticks(x, labels)
axes[1].set_xlabel("Weight decimal places")
axes[1].set_ylabel("QE delta vs. exact weights")
axes[1].set_title("Change in mean quantization error")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


### Margin check: are changed flowers close to a boundary?

For each vector, the margin is the distance to the second-best prototype minus the distance to the best prototype. If each prototype coordinate is rounded to `d` decimals, its movement is bounded by `0.5 × 10⁻ᵈ × sqrt(input_dim)`. Thus a change of winner can only happen when the original margin is below twice that bound.

This is a sufficient risk bound, not a prediction that every at-risk flower will flip.


In [ ]:
prototypes = np.array([
    validation_classifier.get_prototype(x, y)
    for y in range(CANDIDATE_HEIGHT)
    for x in range(CANDIDATE_WIDTH)
])


def _best_and_margin(vectors):
    X = np.asarray(vectors)
    distances = np.linalg.norm(X[:, None, :] - prototypes[None, :, :], axis=2)
    nearest = np.argsort(distances, axis=1)[:, :2]
    rows = np.arange(len(X))
    return nearest[:, 0], distances[rows, nearest[:, 1]] - distances[rows, nearest[:, 0]]


validation_margins = {}
for split_name, vectors in validation_splits.items():
    best, validation_margins[split_name] = _best_and_margin(vectors)
    predicted = _predict_cells(validation_classifier, vectors)
    cells = np.array([
        (p[1] * CANDIDATE_WIDTH + p[0]) if p is not None else -1
        for p in predicted
    ])
    disagreements = int(np.count_nonzero(best != cells))
    print(f"{split_name}: NumPy and classify_features() disagree on "
          f"{disagreements}/{len(vectors)} flowers")


In [ ]:
input_dim = fe.FlowerClassifier.input_dim
print(f"{'dec':>3} {'2*delta':>10} | {'held-out at risk':>17} {'flipped':>8} {'flipped not at risk':>20}")
for row in precision_results:
    two_delta = 10.0 ** -row["decimals"] * np.sqrt(input_dim)
    margins = validation_margins["held_out"]
    at_risk = {int(i) for i in np.flatnonzero(margins < two_delta)}
    flipped = set(row["held_out"]["changed"])
    row["two_delta"] = float(two_delta)
    row["held_out"]["at_risk"] = len(at_risk)
    row["held_out"]["flipped_not_at_risk"] = len(flipped - at_risk)
    print(f"{row['decimals']:>3} {two_delta:>10.3g} | "
          f"{len(at_risk):>9}/{len(margins):<7} {len(flipped):>8} "
          f"{len(flipped - at_risk):>20}")


In [ ]:
eligible_precisions = [
    row for row in precision_results
    if row["held_out"]["changed_rate"] <= MAX_CHANGED_RATE
]
if eligible_precisions:
    # Lowest number of decimals that meets the held-out assignment-drift threshold.
    precision_candidate = min(eligible_precisions, key=lambda row: row["decimals"])
    EMBEDDED_WEIGHT_DECIMALS = precision_candidate["decimals"]
    h = precision_candidate["held_out"]
    print(f"Candidate: {EMBEDDED_WEIGHT_DECIMALS} decimals; "
          f"{len(h['changed'])}/{h['samples']} held-out flowers change cell "
          f"({h['changed_rate']:.2%}); QE delta {h['qe_delta']:+.4g}")
else:
    EMBEDDED_WEIGHT_DECIMALS = max(PRECISIONS)
    print("No tested precision met the held-out drift threshold. "
          f"Using the safest tested setting ({EMBEDDED_WEIGHT_DECIMALS} decimals) provisionally; "
          "review the results before export.")

print("Treat this as a screening heuristic: repeat with multiple SPLIT_SEED values "
      "before relying on a one-percent threshold.")


## 7. Train the production SOM on all flowers

This is the production fit. It intentionally uses all usable flowers, including those held out during precision validation. Because the production model has now seen every flower, its all-data rounding check below is a deployment sanity check, **not** a second held-out estimate.


## 6. Train the real thing, with a training curve

Happy with a grid size? Train it properly on **all** the flowers. One `continue_training()` call trains `FINAL_EPOCHS` epochs of ONE decay, from `BASE_LR`/`BASE_RADIUS` to the end, and returns a `SOMTrainingReport` with a point per epoch: the mean distance of the samples to their species while that epoch ran, measured for free, plus the indices of any flowers it skipped. The classifier is trained in place, so it is the one you keep -- nothing is trained twice.

The prototypes move during an epoch, so that distance differs a little from the quantization error measured after it. For the exact error at a few checkpoints (a full pass over the flowers each) use `evaluation.training_curve_one_decay(final_classifier, samples, FINAL_EPOCHS, BASE_LR, BASE_RADIUS, every)` on a fresh classifier instead. Pass `on_epoch=` to `continue_training()` to follow or stop a long run.

Judge convergence by the error **and** by how many nodes are in use (the error can sit flat while more nodes keep filling).


In [ ]:
FINAL_WIDTH, FINAL_HEIGHT = CANDIDATE_WIDTH, CANDIDATE_HEIGHT
FINAL_EPOCHS = 1500
BASE_LR = 0.9
BASE_RADIUS = max(FINAL_WIDTH, FINAL_HEIGHT) / 2

final_classifier = fe.FlowerClassifier(FINAL_WIDTH, FINAL_HEIGHT, environment)
report = final_classifier.continue_training(samples, FINAL_EPOCHS, BASE_LR, BASE_RADIUS)
print(f'{len(report.skipped)} flowers skipped')
visualize.plot_training_curve(evaluation.epoch_curve(report));


In [ ]:
print('final mean quantization error:', evaluation.mean_quantization_error(final_classifier, samples))
print(evaluation.utilization_summary(final_classifier, samples))

## 8. Qualitative check: what did each node actually learn?

Each cell is one grid node's learned silhouette. Neighboring cells should look like visually similar flower shapes if training organized the space sensibly -- that's the point of a SOM over plain k-means, and it's not really a single number, just look at it.

In [ ]:
visualize.plot_prototype_grid(final_classifier);

In [ ]:
visualize.plot_utilization_heatmap(evaluation.node_utilization(final_classifier, samples));

## 9. Validate and export for embedding

Writes `include/fe/Resources.hpp` with this classifier embedded as `fe::resources::trainedSOM`. Review the diff before committing it -- this overwrites the file. (`tools/som.py --export CLASSIFIER.json --out ../include/fe/Resources.hpp --force` does the same from a saved classifier.)

The weights are rounded to the precision selected by the held-out experiment. The classifier in memory keeps its exact weights. The production-model check immediately before export reports how many assignments change after serialization at that precision; review the warning and the diff before committing.


In [ ]:
# EMBEDDED_WEIGHT_DECIMALS was selected by the held-out precision experiment above.
# Sanity-check the actual production model at the selected precision before export.
production_rounded = _quantized_copy(final_classifier, EMBEDDED_WEIGHT_DECIMALS)
production_exact_predictions = _predict_cells(final_classifier, samples)
production_rounded_predictions = _predict_cells(production_rounded, samples)
production_drift = _quantization_drift(
    production_exact_predictions, production_rounded_predictions,
    final_classifier, production_rounded, samples
)
print(f"Production all-data rounding check: "
      f"{len(production_drift['changed'])}/{production_drift['samples']} cells change "
      f"({production_drift['changed_rate']:.2%}); QE delta {production_drift['qe_delta']:+.4g}")
if production_drift["changed_rate"] > MAX_CHANGED_RATE:
    print("WARNING: production model exceeds the drift threshold; consider more decimals "
          "and rerun this check before committing the embedded resource.")

final_classifier.weight_decimals = EMBEDDED_WEIGHT_DECIMALS
output_path = export.write_resources_hpp(final_classifier.to_json(), '../include/fe/Resources.hpp')
final_classifier.weight_decimals = None
print(f'wrote {output_path} ({output_path.stat().st_size} bytes)')
